In [4]:
import torch
from transformers import PeAudioVideoModel, PeAudioVideoProcessor,AutoProcessor, AutoModel
import time
import numpy as np
from pathlib import Path
import glob
import os
import re
import torch
from natsort import natsorted  


# Base paths
BASE_DIR = Path("/home/amin/Research/Representation/Movie/data/Data/Separated")
OUTPUT_PATH = Path ("/home/amin/Research/Representation/Movie/outputs")

BATCH_SIZE = 16  # Can increase since 2s clips are small
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device is {DEVICE}")

device is cuda


In [5]:
model_id = "facebook/pe-av-small-16-frame"
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"  # 5 minutes

model = PeAudioVideoModel.from_pretrained(model_id).to(DEVICE)
processor = PeAudioVideoProcessor.from_pretrained(model_id)
print("Model & Processor Ready")

KeyboardInterrupt: 

In [ ]:
def extract_part_number(filename):
    """
    Extract part number from filename like 'video_part_001.mp4' or 'audio_part_042.wav'
    Returns the number as int, or None if not found
    """
    match = re.search(r'part[_-]?(\d+)', filename.stem, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None

# Gather all video/audio chunk pairs across all runs
all_videos = []
all_audios = []

for run_idx in range(1, 19):  # Video1 through Video18
    vid_chunk_dir = BASE_DIR / f"Video{run_idx}" / f"Video{run_idx}_chunks"
    aud_chunk_dir = BASE_DIR / f"Audio{run_idx}" / f"Audio{run_idx}_chunks"
    
    if not vid_chunk_dir.exists() or not aud_chunk_dir.exists():
        print(f"Warning: Missing dir for run {run_idx}")
        continue
    
    vids = natsorted(vid_chunk_dir.glob("*.mp4"))
    auds = natsorted(aud_chunk_dir.glob("*.wav"))
    if len(vids) == 0 or len(auds) == 0:
        print(f"Warning: No chunks found for run {run_idx}")
        continue\
    
    # Match by part number
    vid_parts = {}
    for v in vids:
        part_num = extract_part_number(v)
        if part_num is not None:
            vid_parts[part_num] = v
    
    aud_parts = {}
    for a in auds:
        part_num = extract_part_number(a)
        if part_num is not None:
            aud_parts[part_num] = a
    
    # Find common part numbers
    common_parts = sorted(set(vid_parts.keys()) & set(aud_parts.keys()))
    
    for part_num in common_parts:
        all_videos.append(vid_parts[part_num])
        all_audios.append(aud_parts[part_num])
    
    print(f"Run {run_idx}: {len(common_parts)} paired chunks (video: {len(vids)}, audio: {len(auds)})")

paired = list(zip(all_videos, all_audios))
print(f"\n{'='*60}")
print(f"Total: {len(paired)} paired 2s segments")
print(f"{'='*60}\n")

if len(paired) == 0:
    print("ERROR: No paired segments found!")
    exit(1)

Run 1: 106 paired chunks (video: 106, audio: 106)
Run 2: 111 paired chunks (video: 111, audio: 111)
Run 3: 94 paired chunks (video: 94, audio: 94)
Run 4: 32 paired chunks (video: 32, audio: 32)
Run 5: 42 paired chunks (video: 42, audio: 42)
Run 7: 114 paired chunks (video: 114, audio: 114)
Run 8: 130 paired chunks (video: 130, audio: 130)
Run 9: 125 paired chunks (video: 125, audio: 125)
Run 10: 42 paired chunks (video: 42, audio: 42)
Run 11: 91 paired chunks (video: 91, audio: 91)
Run 12: 102 paired chunks (video: 102, audio: 102)
Run 13: 72 paired chunks (video: 72, audio: 72)
Run 14: 42 paired chunks (video: 42, audio: 42)
Run 15: 117 paired chunks (video: 117, audio: 117)
Run 16: 115 paired chunks (video: 115, audio: 115)
Run 17: 128 paired chunks (video: 128, audio: 128)
Run 18: 40 paired chunks (video: 40, audio: 40)

Total: 1503 paired 2s segments



In [ ]:

# Extract embeddings
video_out, audio_out, av_out = [], [], []
clip_names = []
run_ids = []

with torch.inference_mode():
    for i in range(0, len(paired), BATCH_SIZE):
        batch_pairs = paired[i:i + BATCH_SIZE]
        v_batch = [str(v) for v, a in batch_pairs]
        a_batch = [str(a) for v, a in batch_pairs]
        
        inputs = processor(videos=v_batch, audio=a_batch, return_tensors="pt").to(DEVICE)
        
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(**inputs)
        
        video_out.append(outputs.video_embeds.float().cpu())
        audio_out.append(outputs.audio_embeds.float().cpu())
        av_out.append(outputs.audio_video_embeds.float().cpu())
        
        for v, a in batch_pairs:
            clip_names.append(v.stem)
            run_ids.append(v.parent.parent.name)
        
        if (i // BATCH_SIZE) % 50 == 0 or i == 0:
            print(f"Processed {i + len(batch_pairs)}/{len(paired)}")

video_embeds = torch.cat(video_out, dim=0)
audio_embeds = torch.cat(audio_out, dim=0)
av_embeds = torch.cat(av_out, dim=0)

print(f"\nDone!")
print(f"Video embeddings: {video_embeds.shape}")
print(f"Audio embeddings: {audio_embeds.shape}")
print(f"AV embeddings:    {av_embeds.shape}")

Loading weights:   0%|          | 0/859 [00:00<?, ?it/s]

Model loaded


/home/amin/miniconda3/envs/peav/lib/python3.10/site-packages/transformers/video_processing_utils.py:873: UserWarning: `torchcodec` is not installed and cannot be used to decode the video by default. Falling back to `torchvision`. Note that `torchvision` decoding is deprecated and will be removed in future versions. 
  warnings.warn(
/home/amin/miniconda3/envs/peav/lib/python3.10/site-packages/transformers/video_utils.py:536: UserWarning: Using `torchvision` for video decoding is deprecated and will be removed in future versions. Please use `torchcodec` instead.
  warnings.warn(
/home/amin/miniconda3/envs/peav/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  

Processed 16/1503
Processed 816/1503

Done!
Video embeddings: torch.Size([1503, 1024])
Audio embeddings: torch.Size([1503, 1024])
AV embeddings:    torch.Size([1503, 1024])


In [ ]:
torch.save({
    "video": video_embeds,
    "audio": audio_embeds,
    "av": av_embeds,
    "clip_names": clip_names,
    "run_ids": run_ids,
}, OUTPUT_PATH)

print(f"saved to {OUTPUT_PATH}")

Saved to /home/amin/Research/Representation/Movie/data/Data/Separated/peav_small_embeds_2s.pt


In [ ]:
import numpy as np

np.savez(
    OUTPUT_PATH.with_suffix('.npz'),
    video=video_embeds.numpy(),
    audio=audio_embeds.numpy(),
    av=av_embeds.numpy(),
    clip_names=np.array(clip_names),
    run_ids=np.array(run_ids),
)
print(f"also saved to {OUTPUT_PATH.with_suffix('.npz')}")

also saved to /home/amin/Research/Representation/Movie/data/Data/Separated/peav_small_embeds_2s.npz
